# Q-Learning on GridWorld

This notebook walks through Q-Learning from scratch on a custom 8×8 grid environment. By the end you'll have a trained agent, a value heatmap, and a greedy policy visualization.

## The algorithm

Q-Learning maintains a table `Q[state, action]` and updates it after every step:

```
Q[s, a] ← Q[s, a] + α (r + γ · max_a' Q[s', a'] - Q[s, a])
```

The key insight is that we're chasing a *bootstrapped* target — we don't need to wait until the episode ends to learn.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.environments.gridworld import GridWorld
from src.agents.q_learning import QLearningAgent
from src.utils.seeding import set_seed

set_seed(42)
env = GridWorld(size=8)
agent = QLearningAgent(
    n_states=env.n_states,
    n_actions=env.action_space.n,
    alpha=0.1, gamma=0.99,
    epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01,
)
print(f"State space: {env.n_states}  |  Action space: {env.action_space.n}")

In [ ]:
rewards = []
for ep in range(50):
    state, _ = env.reset()
    ep_reward = 0.0
    done = False
    while not done:
        action = agent.choose_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        agent.update(state, action, reward, next_state, done)
        ep_reward += reward
        state = next_state
    rewards.append(ep_reward)

plt.figure(figsize=(10, 3))
plt.plot(rewards, alpha=0.5, label='Episode reward')
rolling = np.convolve(rewards, np.ones(10)/10, mode='valid')
plt.plot(range(9, len(rewards)), rolling, linewidth=2, label='10-ep mean')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('Q-Learning on GridWorld (50 episodes)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Value heatmap
Q = agent.get_q_table()
V = np.max(Q, axis=1).reshape(8, 8)
im = axes[0].imshow(V, cmap='viridis', origin='upper')
plt.colorbar(im, ax=axes[0])
axes[0].plot(7, 7, 'r*', markersize=15, label='Goal')
for r, c in [(2,3),(3,5),(5,2),(6,4)]:
    axes[0].plot(c, r, 'wx', markersize=12)
axes[0].set_title('Max Q per cell (value estimate)')
axes[0].legend()

# Greedy policy
_arrows = {0: (0, -0.35), 1: (0, 0.35), 2: (-0.35, 0), 3: (0.35, 0)}
policy = np.argmax(Q, axis=1).reshape(8, 8)
axes[1].imshow(V, cmap='coolwarm', origin='upper', alpha=0.4)
for r in range(8):
    for c in range(8):
        dx, dy = _arrows[policy[r, c]]
        axes[1].annotate('', xy=(c+dx, r+dy), xytext=(c, r),
                         arrowprops=dict(arrowstyle='->', color='black', lw=1.0))
axes[1].set_title('Greedy policy after 50 episodes')

plt.tight_layout()
plt.show()

After 50 episodes the agent isn't fully converged yet — run more episodes (300 via `python -m src.training.trainer --config-name q_learning_gridworld`) to see the policy sharpen.